# JobSeekr Fresh - Extract Demo

Every code cell starts with an editable JSON block. Change the JSON in that cell, then run the cell.

Cell flow:

1. Start browser
2. Open jobs page
3. Fill keyword and location
4. Open and read filters
5. Sync filters
6. Apply and read listings
7. Read pagination
8. Close browser

In [1]:
import importlib
import browser.driver as browser_driver
import core.config as core_config

browser_driver = importlib.reload(browser_driver)
core_config = importlib.reload(core_config)

import json
import time
from pathlib import Path

from browser.driver import build_driver
from core.config import load_app_config
from core.logging import TreeLogger
from storage.embedded_mongo import EmbeddedMongoStore

# Start browser and shared context.
cell_input_json = r'''
{
  "browser": {
    "headless": false,
    "start_maximized": true
  },
  "profile_key": "browser"
}
'''

cell_input = json.loads(cell_input_json)
app_config = load_app_config()
browser_cfg = dict(app_config['profile']['browser'])
browser_cfg.update(cell_input.get('browser', {}))
store = EmbeddedMongoStore(Path(app_config['profile']['mongo_file']))
logger = TreeLogger('extract')
driver = build_driver(browser_cfg)

print(json.dumps({'browser': browser_cfg, 'profile_key': cell_input['profile_key']}, indent=2))


config: dir D:\_Desktop\Projects\Automations prj\Job_search\JobSeekr_fresh\config
config: load profile.json
config: load extract.json
config: load presets.json
store: open runtime\mongo_store.json
driver: start
driver: launch chrome
driver: ready
{
  "browser": {
    "headless": false,
    "use_stealth": true,
    "version_main": 149,
    "page_load_timeout_seconds": 45,
    "wait_timeout_seconds": 20,
    "start_maximized": true,
    "user_data_dir": "runtime/chrome_user_data"
  },
  "profile_key": "browser"
}


In [ ]:
import importlib
import browser.linkedin_jobs as linkedin_jobs

linkedin_jobs = importlib.reload(linkedin_jobs)

import json

from browser.linkedin_jobs import open_jobs_search_page

# Open the LinkedIn jobs page.
cell_input_json = r'''
{
  "url": "https://www.linkedin.com/jobs/search",
  "delay_seconds": 1.0
}
'''

cell_input = json.loads(cell_input_json)
page_state = open_jobs_search_page(driver, cell_input['url'], delay_seconds=cell_input['delay_seconds'])
print(json.dumps(page_state, indent=2))


In [ ]:
import importlib
import browser.linkedin_jobs as linkedin_jobs

linkedin_jobs = importlib.reload(linkedin_jobs)

import json

from browser.linkedin_jobs import set_keyword_input, set_location_input

# Fill keyword and location.
cell_input_json = r'''
{
  "keyword": "data analyst",
  "location": "dubai, united arab emirates",
  "delay_seconds": 1.0
}
'''

cell_input = json.loads(cell_input_json)
keyword_state = set_keyword_input(driver, cell_input['keyword'], delay_seconds=cell_input['delay_seconds'])
location_state = set_location_input(driver, cell_input['location'], delay_seconds=cell_input['delay_seconds'])
print(json.dumps({'keyword': keyword_state, 'location': location_state}, indent=2))


In [ ]:
import importlib
import browser.linkedin_jobs as linkedin_jobs

linkedin_jobs = importlib.reload(linkedin_jobs)

import json

from browser.linkedin_jobs import open_all_filters_menu, read_filters_state

# Open filters and read the current state.
cell_input_json = r'''
{
  "open_filters": true,
  "delay_seconds": 1.0
}
'''

cell_input = json.loads(cell_input_json)
if cell_input.get('open_filters'):
    print(json.dumps(open_all_filters_menu(driver, delay_seconds=cell_input['delay_seconds']), indent=2))
    time.sleep(cell_input['delay_seconds'])
current_filters = read_filters_state(driver, ensure_open=not cell_input.get('open_filters'))
print(json.dumps(current_filters, indent=2))


In [ ]:
import importlib
import browser.linkedin_jobs as linkedin_jobs

linkedin_jobs = importlib.reload(linkedin_jobs)

import json

from browser.linkedin_jobs import sync_filters_state

# Sync filter choices.
cell_input_json = r'''
{
  "filter_by": "Jobs",
  "filters": [
    {
      "section": "Sort by",
      "type": "radio",
      "input": "Most recent"
    },
    {
      "section": "Date posted",
      "type": "radio",
      "input": "Past week"
    },
    {
      "section": "Experience level",
      "type": "checkbox",
      "inputs": [
        {"name": "Entry level", "state": true}
      ]
    }
  ],
  "delay_seconds": 0.2
}
'''

cell_input = json.loads(cell_input_json)
synced_filters = sync_filters_state(driver, cell_input, delay_seconds=cell_input['delay_seconds'])
print(json.dumps(synced_filters, indent=2))


In [ ]:
import importlib
import browser.linkedin_jobs as linkedin_jobs

linkedin_jobs = importlib.reload(linkedin_jobs)

import json

from browser.linkedin_jobs import show_results

# Apply filters.
cell_input_json = r'''
{
  "apply_filters": true,
  "delay_seconds": 1.0
}
'''

cell_input = json.loads(cell_input_json)
if cell_input.get('apply_filters'):
    print(json.dumps(show_results(driver, delay_seconds=cell_input['delay_seconds']), indent=2))


In [ ]:
import importlib
import browser.linkedin_jobs as linkedin_jobs

linkedin_jobs = importlib.reload(linkedin_jobs)

import json

from browser.linkedin_jobs import extract_listings

# Read listings.
cell_input_json = r'''
{
  "read_listings": true
}
'''

cell_input = json.loads(cell_input_json)
if cell_input.get('read_listings'):
    listings = extract_listings(driver)
    print(json.dumps(listings, indent=2))


In [ ]:
# Read one listing detail.
import importlib
import browser.linkedin_jobs as linkedin_jobs
import parsers.listing_detail as listing_detail_parser
import stages.listing_detail as listing_detail

linkedin_jobs = importlib.reload(linkedin_jobs)
listing_detail_parser = importlib.reload(listing_detail_parser)
listing_detail = importlib.reload(listing_detail)

import json

from stages.listing_detail import extract_listing_detail

cell_input_json = r'''
{
  "index": 8,
  "delay_seconds": 1.0,
  "delay_jitter": 0.2
}
'''

cell_input = json.loads(cell_input_json)
detail_result = extract_listing_detail(
    driver,
    listings,
    index=cell_input["index"],
    delay_seconds=cell_input["delay_seconds"],
    delay_jitter=cell_input["delay_jitter"],
)
print(json.dumps(detail_result, indent=2, ensure_ascii=False))


In [26]:
# Convert the current page to markdown.
import importlib
import parsers.page_markdown as page_markdown
import browser.markdown as markdown_tools
page_markdown = importlib.reload(page_markdown)
markdown_tools = importlib.reload(markdown_tools)
import json
from browser.markdown import output_markdown

cell_input_json = r'''
{
  "preview_chars": 600000000
}
'''

cell_input = json.loads(cell_input_json)
markdown_text, interactables = output_markdown(driver)
print(markdown_text[:cell_input["preview_chars"]])
# print(json.dumps(interactables, indent=2, ensure_ascii=False))

text [[i1]]

Ignore this input please

[DuckDuckGo](https://duckduckgo.com/?t=h_&kp=1) [[i2]]

Search privately [[i3]]

search [disabled] [[i4]]

1 [[i5]]

h_ [[i6]]

[Protection. Privacy. Peace of mind.](https://duckduckgo.com/compare-privacy) [[i7]]

Open menu [[i8]]

- All [[i9]]
- Images [[i10]]
- Videos [[i11]]
- News [[i12]]
- Maps [[i13]]

- Search Assist [[i14]]
- Duck.ai [[i15]]
- Search Settings [[i16]]

Protected

DuckDuckGo never tracks your searches.

[Learn More](https://duckduckgo.com/duckduckgo-help-pages/search-privacy/) [[i17]]

You can hide this reminder in Search Settings [[i18]]

All regions [[i19]]

Search [[i20]]

All regions

Argentina

Australia

Austria

Belgium (fr)

Belgium (nl)

Brazil

Bulgaria

Canada (en)

Canada (fr)

Catalonia

Chile

China

Colombia

Croatia

Czechia

Denmark

Estonia

Finland

France

Germany

Greece

Hong Kong

Hungary

Iceland

India (en)

Indonesia (en)

Ireland

Israel (en)

Italy

Japan

Korea

Latvia

Lithuania

Malaysia (en)



In [ ]:
# Test an interaction diff from the current markdown output.
from importlib import import_module, reload

interact_tools = import_module("browser.interact")
interact_tools = reload(interact_tools)

import json

cell_input_json = r'''
{
  "interaction_type": "click",
  "target_id": "i25",
  "delay_seconds": 1
}
'''

cell_input = json.loads(cell_input_json)
interaction_result = interact_tools.interact(
    driver,
    markdown_text,
    interactables,
    cell_input["interaction_type"],
    cell_input["target_id"],
    delay_seconds=cell_input["delay_seconds"],
)
print(json.dumps(interaction_result, indent=2, ensure_ascii=False))

In [27]:
# Test an interaction diff from the current markdown output.
from importlib import import_module, reload

interact_tools = import_module("browser.interact")
interact_tools = reload(interact_tools)

import json

cell_input_json = r'''
{
  "interaction_type": "input_text",
  "target_id": "i3?value=engineer",
  "delay_seconds": 1
}
'''

cell_input = json.loads(cell_input_json)
interaction_result = interact_tools.interact(
    driver,
    markdown_text,
    interactables,
    cell_input["interaction_type"],
    cell_input["target_id"],
    delay_seconds=cell_input["delay_seconds"],
)
print(json.dumps(interaction_result, indent=2, ensure_ascii=False))

{
  "status": "success",
  "interaction_type": "input_text",
  "requested_target_id": "i3?value=engineer",
  "target_id": "i3",
  "target": {
    "kind": "interactable",
    "id": "i3",
    "order": 3,
    "type": "input",
    "text": "Search privately",
    "anchor_text": "Search privately",
    "aria_label": "",
    "role": "combobox",
    "href": "",
    "value": "",
    "state": {
      "input_type": ""
    },
    "state_text": "",
    "state_signature": "",
    "locator": {
      "kind": "css",
      "value": "#search_form_input"
    },
    "ref": "[[i3]]"
  },
  "payload": {
    "value": "engineer"
  },
  "diffs": {
    "changed": [],
    "added": [
      "clear [[i4]]",
      "search [[i5]]",
      "engineer ing",
      "engineer 's day",
      "engineer ing drawing",
      "engineer ing mathematics",
      "engineer ing khurai thongam leikai rural",
      "engineer ing moirangkampu",
      "engineer s india limited",
      "engineer ing mechanics",
      "engineer"
    ],
    "

In [ ]:
import importlib
import browser.linkedin_jobs as linkedin_jobs

linkedin_jobs = importlib.reload(linkedin_jobs)

import json

from browser.linkedin_jobs import get_visible_pages

# Read pagination state.
cell_input_json = r'''
{
  "read_pages": true
}
'''

cell_input = json.loads(cell_input_json)
if cell_input.get('read_pages'):
    pages = get_visible_pages(driver)
    print(json.dumps({'pages': pages['pages']}, indent=2))


In [ ]:
import importlib
import browser.linkedin_jobs as linkedin_jobs

linkedin_jobs = importlib.reload(linkedin_jobs)

import json

from browser.linkedin_jobs import get_current_page, go_to_page

# Go to a page and verify the current page.
cell_input_json = r'''
{
  "page": 2,
  "delay_seconds": 1.0
}
'''

cell_input = json.loads(cell_input_json)
print(json.dumps(go_to_page(driver, cell_input['page'], delay_seconds=cell_input['delay_seconds']), indent=2))
print(json.dumps(get_current_page(driver), indent=2))


In [ ]:
import importlib
import browser.driver as browser_driver

browser_driver = importlib.reload(browser_driver)

import json

from browser.driver import close_driver

# Close browser.
cell_input_json = r'''
{
  "close_driver": true
}
'''

cell_input = json.loads(cell_input_json)
if cell_input.get('close_driver'):
    close_driver(driver)
    print(json.dumps({'status': 'Driver closed.'}, indent=2))
